In [1]:
import pandas as pd
import os
import psutil
from pathlib import Path

In [2]:
#Memory before loading dataset
#process = psutil.Process(os.getpid())
#mem_before = process.memory_info().rss / (1024 ** 2)
mem_before = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)
print(f"Memory BEFORE: {mem_before: .3f} GB")

Memory BEFORE:  0.104 GB


In [3]:
data_dir = Path("../data/tfl-cycling-journey-data-2024-2025/clean-2024-2025-csv")
csv_files = sorted(data_dir.glob("*.csv"))
print(f"Found {len(csv_files)} files")


Found 48 files


In [4]:
%%time
df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

CPU times: user 20.2 s, sys: 1.65 s, total: 21.9 s
Wall time: 21.9 s


Loading 48 csv files took over 20 seconds (on my system). Will use Polars to speed things up and reduce memory footprint.

In [5]:
print(f"Combined shape: {df.shape}")

Combined shape: (17823393, 11)


In [6]:
display(df.head())

,Number,Start date,Start station number,Start station,End date,End station number,End station,Bike number,Bike model,Total duration,Total duration (ms)
0,136666627,2024-01-14 23:59,1108,"North Wharf Road, Paddington",2024-01-15 00:06,3423.0,"Maida Vale, Maida Vale",53020.0,CLASSIC,6m 47s,407799.0
1,136666625,2024-01-14 23:57,3447,"Gloucester Road (North), Kensington",2024-01-15 00:05,1214.0,"Kensington Olympia Station, Olympia",54559.0,CLASSIC,8m 1s,481276.0
2,136666626,2024-01-14 23:57,1090,"Warren Street Station, Euston",2024-01-15 00:02,200005.0,"New Cavendish Street, Marylebone",52133.0,CLASSIC,5m 11s,311788.0
3,136666622,2024-01-14 23:56,200058,"Northdown Street, King's Cross",2024-01-15 00:06,2687.0,"Brick Lane Market, Shoreditch",60341.0,PBSC_EBIKE,10m 14s,614030.0
4,136666623,2024-01-14 23:56,1052,"Soho Square , Soho",2024-01-15 00:23,1150.0,"Queensway, Kensington Gardens",55212.0,CLASSIC,27m 1s,1621583.0


In [8]:
%%time

#Memory After loading dataset

mem_after = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)
print(f"Memory AFTER: {mem_after: .3f} GB")
print(f"Memory used: {(mem_after - mem_before):.3f} GB" )

Memory AFTER:  2.952 GB
Memory used: 2.848 GB
CPU times: user 1.85 ms, sys: 0 ns, total: 1.85 ms
Wall time: 1.14 ms


In [9]:
%%time
print(df.info())
print(df.describe())
print("Columns:", df.columns.tolist())


<class 'pandas.DataFrame'>
RangeIndex: 17823393 entries, 0 to 17823392
Data columns (total 11 columns):
 #   Column                Dtype  
---  ------                -----  
 0   Number                int64  
 1   Start date            str    
 2   Start station number  int64  
 3   Start station         str    
 4   End date              str    
 5   End station number    float64
 6   End station           str    
 7   Bike number           float64
 8   Bike model            str    
 9   Total duration        str    
 10  Total duration (ms)   float64
dtypes: float64(3), int64(2), str(6)
memory usage: 1.5 GB
None
             Number  Start station number  End station number   Bike number  \
count  1.782339e+07          1.782339e+07        1.782321e+07  1.782339e+07   
mean   1.456048e+08          8.238183e+05        8.136981e+05  5.153370e+04   
std    5.286946e+06          2.912463e+07        2.883097e+07  1.174979e+04   
min    1.364503e+08          9.590000e+02        9.590000e+02 